In [ ]:
# !pip install yfinance
# !pip install TA-Lib
# !pip install numpy
# !pip install pandas
# !pip install vectorbt
# !pip install scipy
# !pip install seaborn

In [ ]:
# Clone repo for lib/ access (Colab only)
import os
if not os.path.exists('/content/trading-strategies'):
    !git clone https://github.com/r-giov/trading-strategies.git /content/trading-strategies
os.chdir('/content/trading-strategies')

In [ ]:
import sys, os
for _p in ['/content/trading-strategies', os.path.join(os.getcwd(), '..')]:
    if os.path.isdir(os.path.join(_p, 'lib')) and _p not in sys.path:
        sys.path.insert(0, _p)

import yfinance as yf
import talib
import numpy as np
import pandas as pd
import vectorbt as vbt
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from itertools import product
import warnings

warnings.filterwarnings('ignore')
pd.set_option('future.no_silent_downcasting', True)
from lib.data_manager import download, download_multi, setup_colab_secrets

In [ ]:
# ════════════════════════════════════════════════════════════════
# CONFIGURATION — RSI Momentum Portfolio
# ════════════════════════════════════════════════════════════════

# FTMO-tradeable universe (only assets that existed before 2020)
UNIVERSE = {
    # Crypto (FTMO confirmed)
    'BTC-USD': 'BTCUSD', 'ETH-USD': 'ETHUSD', 'LTC-USD': 'LTCUSD',
    'XRP-USD': 'XRPUSD', 'ADA-USD': 'ADAUSD', 'DOGE-USD': 'DOGEUSD',
    'LINK-USD': 'LINKUSD', 'SOL-USD': 'SOLUSD', 'AVAX-USD': 'AVAXUSD',
    'DOT-USD': 'DOTUSD', 'BNB-USD': 'BNBUSD',
    # Indices
    '^GSPC': 'SP500', '^DJI': 'US30', '^IXIC': 'US100', '^GDAXI': 'DAX',
    # Commodities
    'GC=F': 'Gold', 'SI=F': 'Silver', 'CL=F': 'CrudeOil',
    # High-beta equities
    'QQQ': 'QQQ', 'NVDA': 'NVDA', 'TSLA': 'TSLA', 'AMD': 'AMD',
    'META': 'META', 'MSTR': 'MSTR', 'COIN': 'COIN',
}

START_DATE = '2019-06-01'  # Ensures 252 bars of warmup before first trade

# ── Core strategy params ──
RSI_PERIOD = 14
TOP_N = 5
REBAL_FREQ = 'M'          # W=weekly, M=monthly, Q=quarterly
RSI_THRESHOLD = 50         # Only buy assets with RSI above this
LOOKBACK_RANK = 21         # Days to average RSI for ranking stability

# ── Risk management params (NEW) ──
REGIME_SMA = 200           # Universe trend filter: EW portfolio above/below 200-day SMA
CIRCUIT_BREAKER_PCT = -0.08  # Go to cash if portfolio drawdown exceeds this (-8%)
COOLDOWN_DAYS = 15         # Stay in cash this many days after circuit breaker triggers
USE_INVVOL_WEIGHTS = True  # Inverse-volatility weighting (vs equal-weight)
VOL_LOOKBACK = 60          # Days for volatility estimation (inv-vol weights)
MAX_CRYPTO_PCT = 0.60      # Max portfolio allocation to crypto assets

# Backtest settings (standard)
INIT_CASH = 100_000
FEES = 0.0005
SLIPPAGE = 0.0005
FREQ = 'D'
TRAIN_RATIO = 0.60

# ── Asset sector tags (for concentration limits) ──
CRYPTO_ASSETS = {'BTCUSD', 'ETHUSD', 'LTCUSD', 'XRPUSD', 'ADAUSD', 'DOGEUSD',
                 'LINKUSD', 'SOLUSD', 'AVAXUSD', 'DOTUSD', 'BNBUSD'}

# Download all universe data
print(f'Downloading {len(UNIVERSE)} instruments...')
_raw = download_multi(list(UNIVERSE.keys()), START_DATE)

# Build close price DataFrame (only assets with enough data)
close_dict = {}
for ticker, name in UNIVERSE.items():
    if ticker in _raw and len(_raw[ticker]) > 252:
        c = _raw[ticker]['Close'].astype(float).squeeze()
        c.name = name
        close_dict[name] = c
        print(f'  OK: {name:12s} ({ticker:12s}) — {len(c)} bars, from {c.index[0].date()}')
    else:
        print(f'  SKIP: {name} — insufficient data')

# Align all series to common dates
close_df = pd.DataFrame(close_dict)
print(f'\nUniverse: {close_df.shape[1]} assets, {close_df.shape[0]} trading days')
print(f'Date range: {close_df.index[0].date()} to {close_df.index[-1].date()}')
print(f'Assets with >10% missing: {(close_df.isna().mean() > 0.10).sum()}')

In [ ]:
# ════════════════════════════════════════════════════════════════
# RSI MOMENTUM PORTFOLIO ENGINE (v2 — with risk management)
# ════════════════════════════════════════════════════════════════

def run_rsi_portfolio(close_df, rsi_period=14, top_n=5, rebal_freq='M',
                      rsi_threshold=50, lookback_rank=21,
                      regime_sma=200, circuit_breaker_pct=-0.08,
                      cooldown_days=15, use_invvol=True, vol_lookback=60,
                      max_crypto_pct=0.60, crypto_assets=None,
                      start_date=None, end_date=None):
    """
    RSI Momentum Portfolio v2 with risk management:
    
    1. REGIME FILTER: Equal-weight universe must be above its SMA to go long.
       Below SMA → hold cash (avoids bear markets like 2022).
    2. CIRCUIT BREAKER: If portfolio drawdown from peak exceeds threshold,
       go to 100% cash for cooldown_days. Directly maps to FTMO max-loss rule.
    3. INVERSE-VOL WEIGHTING: Weight positions by 1/volatility instead of
       equal-weight. Automatically reduces allocation to crashing assets.
    4. CRYPTO CONCENTRATION CAP: Limits total crypto allocation to max_crypto_pct.
    
    Returns: daily portfolio returns Series, holdings history list
    """
    df = close_df.copy()
    if start_date:
        df = df[df.index >= start_date]
    if end_date:
        df = df[df.index <= end_date]
    
    if crypto_assets is None:
        crypto_assets = set()
    
    # Compute RSI for all assets
    rsi_df = pd.DataFrame(index=df.index)
    for col in df.columns:
        prices = df[col].dropna()
        if len(prices) > rsi_period:
            rsi_vals = talib.RSI(prices.values, timeperiod=rsi_period)
            rsi_series = pd.Series(rsi_vals, index=prices.index, name=col)
            rsi_df[col] = rsi_series
    
    # Smoothed RSI for ranking stability
    rsi_smooth = rsi_df.rolling(lookback_rank, min_periods=1).mean()
    
    # Daily returns for all assets
    returns_df = df.pct_change()
    
    # ── REGIME FILTER: equal-weight universe SMA ──
    ew_index = df.divide(df.iloc[0]).mean(axis=1)  # normalized EW index
    ew_sma = ew_index.rolling(regime_sma, min_periods=regime_sma).mean()
    regime_bull = ew_index > ew_sma  # True = risk-on, False = risk-off
    
    # Rolling volatility for inverse-vol weighting
    asset_vol = returns_df.rolling(vol_lookback, min_periods=20).std() * np.sqrt(252)
    
    # Generate rebalance dates
    if rebal_freq == 'W':
        rebal_dates = df.resample('W-FRI').last().index
    elif rebal_freq == 'M':
        rebal_dates = df.resample('ME').last().index
    elif rebal_freq == 'Q':
        rebal_dates = df.resample('QE').last().index
    else:
        rebal_dates = df.resample('ME').last().index
    
    # Skip dates before we have enough history
    warmup = max(rsi_period + lookback_rank + 5, regime_sma + 5, vol_lookback + 5)
    rebal_dates = rebal_dates[rebal_dates >= df.index[min(warmup, len(df) - 1)]]
    
    portfolio_returns = []
    holdings_history = []
    
    # ── CIRCUIT BREAKER state ──
    cumulative_equity = 1.0
    peak_equity = 1.0
    cooldown_until = None  # date when cooldown expires
    
    for i in range(len(rebal_dates) - 1):
        rebal_date = rebal_dates[i]
        next_rebal = rebal_dates[i + 1]
        
        # ── CHECK CIRCUIT BREAKER ──
        if cooldown_until is not None and rebal_date < cooldown_until:
            # Still in cooldown — stay in cash
            period_mask = (returns_df.index > rebal_date) & (returns_df.index <= next_rebal)
            cash_rets = pd.Series(0.0, index=returns_df.index[period_mask])
            portfolio_returns.append(cash_rets)
            holdings_history.append({
                'date': rebal_date, 'holdings': ['CASH (circuit breaker)'],
                'n_selected': 0, 'top_rsi': None, 'regime': 'COOLDOWN',
                'weights': {}
            })
            continue
        
        # ── CHECK REGIME FILTER ──
        if rebal_date in regime_bull.index and not regime_bull.loc[rebal_date]:
            # Bear regime — go to cash
            period_mask = (returns_df.index > rebal_date) & (returns_df.index <= next_rebal)
            cash_rets = pd.Series(0.0, index=returns_df.index[period_mask])
            portfolio_returns.append(cash_rets)
            # Still track equity for circuit breaker
            holdings_history.append({
                'date': rebal_date, 'holdings': ['CASH (regime filter)'],
                'n_selected': 0, 'top_rsi': None, 'regime': 'BEAR',
                'weights': {}
            })
            continue
        
        # Get RSI scores as of rebalance date (no lookahead)
        if rebal_date not in rsi_smooth.index:
            continue
        scores = rsi_smooth.loc[rebal_date].dropna()
        
        # Filter: only assets above RSI threshold
        above_thresh = scores[scores >= rsi_threshold].sort_values(ascending=False)
        
        # Select top N
        selected = above_thresh.head(top_n)
        
        if len(selected) == 0:
            period_mask = (returns_df.index > rebal_date) & (returns_df.index <= next_rebal)
            cash_rets = pd.Series(0.0, index=returns_df.index[period_mask])
            portfolio_returns.append(cash_rets)
            holdings_history.append({
                'date': rebal_date, 'holdings': ['CASH'], 'n_selected': 0,
                'top_rsi': None, 'regime': 'BULL', 'weights': {}
            })
            continue
        
        selected_assets = selected.index.tolist()
        
        # ── COMPUTE WEIGHTS ──
        if use_invvol and rebal_date in asset_vol.index:
            vols = asset_vol.loc[rebal_date, selected_assets].dropna()
            vols = vols[vols > 0]
            if len(vols) >= 1:
                # Inverse volatility
                inv_vol = 1.0 / vols
                weights = inv_vol / inv_vol.sum()
                # Re-filter to only assets with valid weights
                selected_assets = weights.index.tolist()
            else:
                weights = pd.Series(1.0 / len(selected_assets),
                                    index=selected_assets)
        else:
            weights = pd.Series(1.0 / len(selected_assets),
                                index=selected_assets)
        
        # ── CRYPTO CONCENTRATION CAP ──
        crypto_in_portfolio = [a for a in selected_assets if a in crypto_assets]
        crypto_weight = weights[crypto_in_portfolio].sum() if crypto_in_portfolio else 0
        
        if crypto_weight > max_crypto_pct and crypto_in_portfolio:
            # Scale down crypto weights proportionally
            scale = max_crypto_pct / crypto_weight
            non_crypto = [a for a in selected_assets if a not in crypto_assets]
            
            for a in crypto_in_portfolio:
                weights[a] *= scale
            
            # Redistribute excess to non-crypto (if any)
            excess = 1.0 - weights.sum()
            if non_crypto and excess > 0:
                for a in non_crypto:
                    weights[a] += excess / len(non_crypto)
            
            # Renormalize
            weights = weights / weights.sum()
        
        # 1-bar execution delay: use returns from day AFTER rebalance
        period_mask = (returns_df.index > rebal_date) & (returns_df.index <= next_rebal)
        period_rets = returns_df.loc[period_mask, selected_assets].fillna(0)
        
        # Weighted daily portfolio return
        daily_port_ret = (period_rets * weights).sum(axis=1)
        
        # Subtract transaction costs at rebalance
        if len(daily_port_ret) > 0:
            daily_port_ret.iloc[0] -= FEES * 2
        
        # ── UPDATE CIRCUIT BREAKER tracking (daily) ──
        for day_idx in range(len(daily_port_ret)):
            cumulative_equity *= (1 + daily_port_ret.iloc[day_idx])
            peak_equity = max(peak_equity, cumulative_equity)
            current_dd = (cumulative_equity - peak_equity) / peak_equity
            
            if current_dd < circuit_breaker_pct:
                # TRIGGER: go to cash, zero out remaining days in this period
                trigger_date = daily_port_ret.index[day_idx]
                cooldown_until = trigger_date + pd.Timedelta(days=cooldown_days)
                # Zero out returns after trigger
                daily_port_ret.iloc[day_idx + 1:] = 0.0
                break
        
        portfolio_returns.append(daily_port_ret)
        holdings_history.append({
            'date': rebal_date,
            'holdings': selected_assets,
            'n_selected': len(selected_assets),
            'top_rsi': float(selected.iloc[0]),
            'regime': 'BULL',
            'weights': weights.to_dict(),
        })
    
    if not portfolio_returns:
        return pd.Series(dtype=float), []
    
    port_daily = pd.concat(portfolio_returns).sort_index()
    port_daily = port_daily[~port_daily.index.duplicated(keep='first')]
    
    return port_daily, holdings_history


def compute_portfolio_metrics(daily_returns, label=''):
    """Compute standard metrics from daily return series."""
    if len(daily_returns) < 10:
        return {}
    cum = (1 + daily_returns).cumprod()
    total_ret = cum.iloc[-1] - 1
    years = (daily_returns.index[-1] - daily_returns.index[0]).days / 365.25
    ann_ret = (1 + total_ret) ** (1 / max(years, 0.01)) - 1
    ann_vol = daily_returns.std() * np.sqrt(252)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else 0
    down = daily_returns[daily_returns < 0]
    down_vol = down.std() * np.sqrt(252) if len(down) > 0 else 0
    sortino = ann_ret / down_vol if down_vol > 0 else 0
    peak = cum.cummax()
    dd = (cum - peak) / peak
    max_dd = dd.min()
    calmar = ann_ret / abs(max_dd) if abs(max_dd) > 1e-9 else 0
    win_rate = (daily_returns > 0).sum() / len(daily_returns) * 100
    
    # Count cash days (0 return)
    cash_days = (daily_returns == 0).sum()
    cash_pct = cash_days / len(daily_returns) * 100
    
    return {
        'label': label,
        'total_return': total_ret,
        'ann_return': ann_ret,
        'ann_volatility': ann_vol,
        'sharpe': sharpe,
        'sortino': sortino,
        'max_dd': max_dd,
        'calmar': calmar,
        'win_rate': win_rate,
        'cash_pct': cash_pct,
        'years': years,
        'n_days': len(daily_returns),
    }

print('Portfolio engine v2 defined (regime filter + circuit breaker + inv-vol weights).')

In [ ]:
# IS/OOS SPLIT
split_idx = int(len(close_df) * TRAIN_RATIO)
split_date = close_df.index[split_idx]
is_start = close_df.index[0]
is_end = split_date
oos_start = split_date
oos_end = close_df.index[-1]

print(f'Full sample:   {close_df.index[0].date()} to {close_df.index[-1].date()} ({len(close_df)} bars)')
print(f'In-sample:     {is_start.date()} to {is_end.date()} ({split_idx} bars)')
print(f'Out-of-sample: {oos_start.date()} to {oos_end.date()} ({len(close_df) - split_idx} bars)')

## RSI Momentum Portfolio Strategy (v2 — Risk-Managed)

**Signal**: Rank all FTMO-tradeable assets by smoothed RSI. Buy the top N assets above the RSI threshold. Inverse-volatility weighted. Rebalance periodically.

**Risk Management Layers**:
1. **Regime Filter (200-SMA)**: Equal-weight universe must be above its 200-day SMA. Below → 100% cash. Avoids prolonged bear markets.
2. **Circuit Breaker (-8% DD)**: If portfolio drawdown from peak exceeds -8%, go to cash for 15 days. Maps directly to FTMO's 10% max-loss rule with a safety buffer.
3. **Inverse-Volatility Weighting**: High-vol assets get smaller positions. Automatically de-risks during volatility spikes.
4. **Crypto Concentration Cap (60%)**: Prevents the portfolio from becoming an all-crypto bet.

**Anti-lookahead measures**:
- RSI computed only on data available at each rebalance date
- Smoothed RSI ranking (rolling mean) to reduce whipsaw
- 1-bar execution delay on rebalance signals
- Regime filter uses lagged SMA (no future data)
- Circuit breaker reacts to realized drawdown only

In [ ]:
# STRATEGY PARAMETERS

print('RSI Momentum Portfolio v2 — Parameters')
print('=' * 55)
print(f'  ── Core ──')
print(f'  RSI Period:          {RSI_PERIOD}')
print(f'  Top N Holdings:      {TOP_N}')
print(f'  Rebalance Freq:      {REBAL_FREQ}')
print(f'  RSI Threshold:       {RSI_THRESHOLD}')
print(f'  Lookback (rank):     {LOOKBACK_RANK} days')
print(f'')
print(f'  ── Risk Management ──')
print(f'  Regime SMA:          {REGIME_SMA}-day (EW universe)')
print(f'  Circuit Breaker:     {CIRCUIT_BREAKER_PCT*100:.0f}% drawdown → cash')
print(f'  Cooldown:            {COOLDOWN_DAYS} days after trigger')
print(f'  Weighting:           {"Inverse-Vol" if USE_INVVOL_WEIGHTS else "Equal-Weight"}')
print(f'  Vol Lookback:        {VOL_LOOKBACK} days')
print(f'  Max Crypto:          {MAX_CRYPTO_PCT*100:.0f}%')
print(f'')
print(f'  ── Backtest ──')
print(f'  Init Cash:           ${INIT_CASH:,.0f}')
print(f'  Fees:                {FEES*100:.2f}%')
print(f'  Slippage:            {SLIPPAGE*100:.2f}%')
print('=' * 55)

In [ ]:
# ════════════════════════════════════════════════════════════════
# IS / OOS CONSISTENCY CHECK (no optimization — same params both halves)
# ════════════════════════════════════════════════════════════════

_risk_params = dict(
    regime_sma=REGIME_SMA, circuit_breaker_pct=CIRCUIT_BREAKER_PCT,
    cooldown_days=COOLDOWN_DAYS, use_invvol=USE_INVVOL_WEIGHTS,
    vol_lookback=VOL_LOOKBACK, max_crypto_pct=MAX_CRYPTO_PCT,
    crypto_assets=CRYPTO_ASSETS,
)

print(f'Running IS/OOS consistency check...')
print(f'  In-sample:     {is_start.date()} to {is_end.date()}')
print(f'  Out-of-sample: {oos_start.date()} to {oos_end.date()}')
print()

# Run on IS
port_is, holdings_is = run_rsi_portfolio(
    close_df, rsi_period=RSI_PERIOD, top_n=TOP_N, rebal_freq=REBAL_FREQ,
    rsi_threshold=RSI_THRESHOLD, lookback_rank=LOOKBACK_RANK,
    start_date=is_start, end_date=is_end, **_risk_params
)
m_is = compute_portfolio_metrics(port_is, 'IS')

# Run on OOS
port_oos, holdings_oos = run_rsi_portfolio(
    close_df, rsi_period=RSI_PERIOD, top_n=TOP_N, rebal_freq=REBAL_FREQ,
    rsi_threshold=RSI_THRESHOLD, lookback_rank=LOOKBACK_RANK,
    start_date=oos_start, end_date=oos_end, **_risk_params
)
m_oos = compute_portfolio_metrics(port_oos, 'OOS')

# Regime stats
is_cash = [h for h in holdings_is if 'CASH' in str(h['holdings'])]
oos_cash = [h for h in holdings_oos if 'CASH' in str(h['holdings'])]
print(f'  IS:  {len(holdings_is)} rebalances, {len(is_cash)} in cash ({len(is_cash)/max(len(holdings_is),1)*100:.0f}%)')
print(f'  OOS: {len(holdings_oos)} rebalances, {len(oos_cash)} in cash ({len(oos_cash)/max(len(holdings_oos),1)*100:.0f}%)')
print()

# Consistency verdict
sharpe_diff = abs(m_is.get('sharpe', 0) - m_oos.get('sharpe', 0))
oos_sharpe = m_oos.get('sharpe', 0)
is_sharpe = m_is.get('sharpe', 0)

print('=' * 80)
print('IS vs OOS CONSISTENCY CHECK')
print('=' * 80)
print(f'{"Metric":<24} {"IS":>12} {"OOS":>12} {"Diff":>12}')
print('-' * 60)
for key, fmt in [('total_return', '.2%'), ('ann_return', '.2%'), ('sharpe', '.3f'),
                  ('sortino', '.3f'), ('max_dd', '.2%'), ('calmar', '.2f'),
                  ('win_rate', '.1f'), ('cash_pct', '.1f')]:
    v_is = m_is.get(key, 0)
    v_oos = m_oos.get(key, 0)
    diff = v_oos - v_is
    suffix = '%' if key in ('win_rate', 'cash_pct') else ''
    print(f'{key:<24} {v_is:>12{fmt}}{suffix} {v_oos:>12{fmt}}{suffix} {diff:>+12{fmt}}{suffix}')
print('=' * 80)

if oos_sharpe > 0.5 and sharpe_diff < 0.5:
    print('✅ CONSISTENT — OOS Sharpe healthy, IS/OOS gap is small')
elif oos_sharpe > 0 and sharpe_diff < 1.0:
    print('⚠️ ACCEPTABLE — OOS positive but some decay from IS')
else:
    print('❌ INCONSISTENT — significant IS/OOS divergence, params may not generalize')

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# FULL-SAMPLE RUN + PERFORMANCE METRICS
# ════════════════════════════════════════════════════════════════════════

b_rsi = RSI_PERIOD; b_topn = TOP_N; b_rebal = REBAL_FREQ; b_thresh = RSI_THRESHOLD

# Full sample
port_full, holdings_full = run_rsi_portfolio(
    close_df, rsi_period=b_rsi, top_n=b_topn, rebal_freq=b_rebal,
    rsi_threshold=b_thresh, lookback_rank=LOOKBACK_RANK, **_risk_params
)

# Buy & Hold benchmark (equal-weight all assets)
bh_rets = close_df.pct_change().mean(axis=1).dropna()
bh_rets = bh_rets.reindex(port_full.index).fillna(0)

m_full = compute_portfolio_metrics(port_full, 'Full')
m_bh = compute_portfolio_metrics(bh_rets, 'B&H')

# Print comparison table
print('=' * 85)
print(f'RSI MOMENTUM PORTFOLIO v2 — RSI({b_rsi}) Top{b_topn} {b_rebal} Thresh={b_thresh}')
print(f'Risk Mgmt: {REGIME_SMA}-SMA regime | {CIRCUIT_BREAKER_PCT*100:.0f}% circuit breaker | Inv-Vol weights | {MAX_CRYPTO_PCT*100:.0f}% crypto cap')
print(f'Period: {port_full.index[0].date()} to {port_full.index[-1].date()}')
print('=' * 85)
print(f'{"Metric":<24} {"Full":>12} {"IS":>12} {"OOS":>12} {"B&H":>12}')
print('-' * 72)
for key, fmt in [('total_return', '.2%'), ('ann_return', '.2%'), ('sharpe', '.3f'),
                  ('sortino', '.3f'), ('max_dd', '.2%'), ('calmar', '.2f'),
                  ('ann_volatility', '.2%'), ('win_rate', '.1f'), ('cash_pct', '.1f')]:
    vals = [m_full.get(key, 0), m_is.get(key, 0), m_oos.get(key, 0), m_bh.get(key, 0)]
    suffix = '%' if key in ('win_rate', 'cash_pct') else ''
    print(f'{key:<24} {vals[0]:>12{fmt}}{suffix} {vals[1]:>12{fmt}}{suffix} {vals[2]:>12{fmt}}{suffix} {vals[3]:>12{fmt}}{suffix}')
print('=' * 85)

# Regime/circuit breaker summary
n_bull = sum(1 for h in holdings_full if h.get('regime') == 'BULL' and h['n_selected'] > 0)
n_bear = sum(1 for h in holdings_full if h.get('regime') == 'BEAR')
n_cooldown = sum(1 for h in holdings_full if h.get('regime') == 'COOLDOWN')
n_cash_other = sum(1 for h in holdings_full if h['n_selected'] == 0 and h.get('regime') == 'BULL')
print(f'\nRegime Summary: {n_bull} invested | {n_bear} bear (cash) | {n_cooldown} circuit breaker | {n_cash_other} no-signal cash')

# ── Compute derived data for dashboard ──
returns = port_full
cum_returns = (1 + returns).cumprod() - 1
equity = (1 + returns).cumprod() * INIT_CASH
running_max = equity.cummax()
drawdown = (equity - running_max) / running_max * 100

monthly_rets = returns.resample('M').apply(lambda x: (1 + x).prod() - 1) * 100
monthly_data = pd.DataFrame({'year': monthly_rets.index.year, 'month': monthly_rets.index.month, 'return': monthly_rets.values})
pivot = monthly_data.pivot_table(values='return', index='month', columns='year')

yearly_rets = returns.resample('YE').apply(lambda x: (1 + x).prod() - 1) * 100

rolling_sharpe = returns.rolling(252).mean() / returns.rolling(252).std() * np.sqrt(252)
rolling_vol = returns.rolling(252).std() * np.sqrt(252) * 100

bh_cum = (1 + bh_rets).cumprod() - 1

# ── Asset contribution ──
asset_total_ret = {}
returns_df_all = close_df.pct_change()
for h_idx in range(len(holdings_full) - 1):
    h = holdings_full[h_idx]
    if h['n_selected'] == 0:
        continue
    next_date = holdings_full[h_idx + 1]['date'] if h_idx + 1 < len(holdings_full) else port_full.index[-1]
    period_mask = (returns_df_all.index > h['date']) & (returns_df_all.index <= next_date)
    weights = h.get('weights', {})
    for asset in h['holdings']:
        if asset.startswith('CASH') or asset not in returns_df_all.columns:
            continue
        w = weights.get(asset, 1.0 / h['n_selected'])
        asset_ret = returns_df_all.loc[period_mask, asset].fillna(0).sum() * w
        asset_total_ret[asset] = asset_total_ret.get(asset, 0) + asset_ret

total_port_ret = sum(asset_total_ret.values())
asset_pct = {k: v / total_port_ret * 100 for k, v in asset_total_ret.items()} if total_port_ret > 0 else {}
asset_pct_sorted = dict(sorted(asset_pct.items(), key=lambda x: x[1], reverse=True))

print(f'\nTop contributors:')
for name, pct in list(asset_pct_sorted.items())[:10]:
    print(f'  {name:12s}: {pct:+.1f}%')

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# PERFORMANCE DASHBOARD — v2 with regime overlay
# ════════════════════════════════════════════════════════════════════════

fig = plt.figure(figsize=(22, 30))
fig.suptitle(f'RSI Momentum Top-{b_topn} Portfolio v2 (Risk-Managed) - Performance Dashboard',
             fontsize=20, fontweight='bold', y=0.995)

gs = gridspec.GridSpec(5, 6, figure=fig, hspace=0.40, wspace=0.45,
                       top=0.97, bottom=0.03, left=0.06, right=0.97)

# ── Regime data for shading ──
ew_index = close_df.divide(close_df.iloc[0]).mean(axis=1)
ew_sma_plot = ew_index.rolling(REGIME_SMA, min_periods=REGIME_SMA).mean()
regime_bear_mask = ew_index < ew_sma_plot

def shade_bear_regime(ax, mask, alpha=0.08):
    """Add red shading for bear regime periods."""
    in_bear = False
    start = None
    for dt, is_bear in mask.items():
        if is_bear and not in_bear:
            start = dt; in_bear = True
        elif not is_bear and in_bear:
            ax.axvspan(start, dt, color='red', alpha=alpha)
            in_bear = False
    if in_bear:
        ax.axvspan(start, mask.index[-1], color='red', alpha=alpha)

# ── Row 1: Cumulative Returns + Key Metrics ──
ax1 = fig.add_subplot(gs[0, :5])
ax1.fill_between(cum_returns.index, cum_returns.values * 100, alpha=0.25, color='#2ca02c')
ax1.plot(cum_returns.index, cum_returns.values * 100, color='#2ca02c', linewidth=1.8)
shade_bear_regime(ax1, regime_bear_mask, alpha=0.12)
ax1.set_title('Cumulative Returns (Net) — Red shading = bear regime (cash)', fontsize=14, fontweight='bold')
ax1.set_ylabel('Return (%)', fontsize=12)
ax1.grid(True, alpha=0.3)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}%'))

metrics_text = (
    f'KEY METRICS\n'
    f'Total Return: {m_full["total_return"]*100:>8.1f}%\n'
    f'CAGR:         {m_full["ann_return"]*100:>8.1f}%\n'
    f'Sharpe:       {m_full["sharpe"]:>8.2f}\n'
    f'Sortino:      {m_full["sortino"]:>8.2f}\n'
    f'Max DD:       {m_full["max_dd"]*100:>7.1f}%\n'
    f'Win Rate:     {m_full["win_rate"]:>7.1f}%\n'
    f'Cash %:       {m_full["cash_pct"]:>7.1f}%'
)
ax_m = fig.add_subplot(gs[0, 5])
ax_m.axis('off')
ax_m.text(0.05, 0.95, metrics_text, transform=ax_m.transAxes,
          fontsize=12, verticalalignment='top', fontfamily='monospace',
          bbox=dict(boxstyle='round,pad=0.6', facecolor='white', edgecolor='#333', alpha=0.95))

# ── Row 2: Drawdown + Monthly Heatmap ──
ax2 = fig.add_subplot(gs[1, :4])
ax2.fill_between(drawdown.index, drawdown.values, alpha=0.45, color='#8b0000')
ax2.plot(drawdown.index, drawdown.values, color='#8b0000', linewidth=0.8)
shade_bear_regime(ax2, regime_bear_mask, alpha=0.12)
worst_idx = drawdown.idxmin()
ax2.scatter([worst_idx], [drawdown.min()], color='red', s=100, zorder=5, edgecolors='black')
ax2.axhline(y=CIRCUIT_BREAKER_PCT * 100, color='orange', linestyle='--', linewidth=1.5,
            alpha=0.8, label=f'Circuit breaker ({CIRCUIT_BREAKER_PCT*100:.0f}%)')
ax2.set_title('Drawdown Over Time', fontsize=15, fontweight='bold')
ax2.set_ylabel('Drawdown (%)', fontsize=12)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

ax3 = fig.add_subplot(gs[1, 4:])
month_labels = ['J','F','M','A','M','J','J','A','S','O','N','D']
y_labels = [month_labels[m-1] for m in pivot.index]
sns.heatmap(pivot, cmap='RdYlGn', center=0, annot=True, fmt='.0f',
            ax=ax3, cbar_kws={'label': 'Return (%)', 'shrink': 0.8},
            yticklabels=y_labels, linewidths=0.5, linecolor='white')
ax3.set_title('Monthly Returns (%)', fontsize=15, fontweight='bold')
ax3.set_ylabel('month', fontsize=11)
ax3.set_xlabel('year', fontsize=11)

# ── Row 3: Asset Attribution + Yearly Results ──
ax4 = fig.add_subplot(gs[2, :3])
top20 = dict(list(asset_pct_sorted.items())[:20])
names = list(reversed(list(top20.keys())))
vals = list(reversed(list(top20.values())))
colors_bar = ['#2ca02c' if v > 0 else '#d62728' for v in vals]
ax4.barh(names, vals, color=colors_bar, alpha=0.85, edgecolor='white', linewidth=0.5)
for i, (n, v) in enumerate(zip(names, vals)):
    ax4.text(v + 0.3, i, f'{v:.1f}%', va='center', fontsize=8, fontweight='bold')
ax4.set_title('Top 20 Asset Performance Attribution', fontsize=15, fontweight='bold')
ax4.set_xlabel('Total Contribution to Returns (%)', fontsize=11)
ax4.grid(True, alpha=0.2, axis='x')

ax5 = fig.add_subplot(gs[2, 3:])
years = yearly_rets.index.year
colors_yr = ['#2ca02c' if v >= 0 else '#d62728' for v in yearly_rets.values]
ax5.bar(years, yearly_rets.values, color=colors_yr, alpha=0.85, edgecolor='darkgreen', linewidth=0.5)
if len(yearly_rets) > 1:
    complete_years = yearly_rets.iloc[:-1]
    mean_ret = complete_years.mean()
    ax5.axhline(y=mean_ret, color='blue', linestyle='--', linewidth=1.8,
                label=f'Mean (complete years): {mean_ret:.1f}%')
    ax5.legend(fontsize=10)
ax5.set_title('Yearly Results', fontsize=15, fontweight='bold')
ax5.set_ylabel('Return (%)', fontsize=12)
ax5.grid(True, alpha=0.3, axis='y')

# ── Row 4: Returns Distribution + Rolling Sharpe + Rolling Vol ──
ax6 = fig.add_subplot(gs[3, :2])
daily_rets_pct = returns[returns != 0].dropna() * 100  # exclude cash days for distribution
ax6.hist(daily_rets_pct, bins=60, density=True, alpha=0.7, color='steelblue', edgecolor='white', linewidth=0.3)
x_norm = np.linspace(daily_rets_pct.min(), daily_rets_pct.max(), 100)
ax6.plot(x_norm, stats.norm.pdf(x_norm, daily_rets_pct.mean(), daily_rets_pct.std()),
         'r--', linewidth=2, label='Normal')
ax6.set_title('Daily Returns Distribution (excl. cash days)', fontsize=14, fontweight='bold')
ax6.set_xlabel('Daily Return (%)', fontsize=12)
ax6.set_ylabel('Density', fontsize=12)
ax6.legend(fontsize=11)
ax6.grid(True, alpha=0.3)

ax7 = fig.add_subplot(gs[3, 2:4])
ax7.plot(rolling_sharpe.index, rolling_sharpe.values, color='purple', linewidth=1)
shade_bear_regime(ax7, regime_bear_mask, alpha=0.12)
mean_rs = rolling_sharpe.dropna().mean()
ax7.axhline(y=mean_rs, color='blue', linestyle='--', linewidth=1.8,
            label=f'Mean: {mean_rs:.2f}')
ax7.axhline(y=0, color='black', linewidth=0.5, alpha=0.3)
ax7.set_title('Rolling Sharpe (252 days)', fontsize=15, fontweight='bold')
ax7.set_ylabel('Sharpe Ratio', fontsize=12)
ax7.legend(fontsize=11)
ax7.grid(True, alpha=0.3)

ax8 = fig.add_subplot(gs[3, 4:])
ax8.plot(rolling_vol.index, rolling_vol.values, color='#ff8c00', linewidth=1)
shade_bear_regime(ax8, regime_bear_mask, alpha=0.12)
ax8.set_title('Rolling Volatility (252 days)', fontsize=15, fontweight='bold')
ax8.set_ylabel('Annualized Volatility (%)', fontsize=12)
ax8.grid(True, alpha=0.3)

# ── Row 5: Strategy vs B&H + Holdings count ──
ax9 = fig.add_subplot(gs[4, :3])
common_idx = cum_returns.index.intersection(bh_cum.index)
ax9.plot(common_idx, cum_returns.loc[common_idx].values * 100, color='#2ca02c',
         linewidth=1.8, label='Strategy')
ax9.plot(common_idx, bh_cum.loc[common_idx].values * 100, color='gray',
         linewidth=1.5, alpha=0.7, label='Buy & Hold (EW Universe)', linestyle='--')
shade_bear_regime(ax9, regime_bear_mask, alpha=0.12)
ax9.axvline(x=split_date, color='red', linestyle=':', alpha=0.5, label='IS/OOS Split')
ax9.set_title('Strategy vs Buy & Hold', fontsize=15, fontweight='bold')
ax9.set_ylabel('Return (%)', fontsize=12)
ax9.legend(fontsize=10)
ax9.grid(True, alpha=0.3)

# Holdings count over time (color-coded by regime)
ax10 = fig.add_subplot(gs[4, 3:])
hold_dates = [h['date'] for h in holdings_full]
hold_counts = [h['n_selected'] for h in holdings_full]
hold_colors = ['#d62728' if h.get('regime') in ('BEAR', 'COOLDOWN') else '#1f77b4'
               for h in holdings_full]
ax10.bar(hold_dates, hold_counts, width=20, color=hold_colors, alpha=0.7)
ax10.axhline(y=np.mean(hold_counts), color='blue', linestyle='--', linewidth=1.5,
             label=f'Avg: {np.mean(hold_counts):.1f}')
ax10.axvline(x=split_date, color='red', linestyle=':', alpha=0.5)
ax10.set_title('Holdings Count (red = cash/regime off)', fontsize=14, fontweight='bold')
ax10.set_ylabel('# Assets Held', fontsize=12)
ax10.legend(fontsize=10)
ax10.grid(True, alpha=0.3, axis='y')

plt.show()

dashboard_fig = fig
print(f'\nDashboard generated for RSI({b_rsi}) Top{b_topn} {b_rebal} Thresh={b_thresh} [v2 risk-managed]')

In [ ]:
# MONTE CARLO SIMULATION — FTMO CHALLENGE

print('=' * 70)
print('\U0001f3b2 MONTE CARLO SIMULATION — FTMO CHALLENGE FEASIBILITY')
print('=' * 70)

daily_rets = port_full.values.ravel()
daily_rets = daily_rets[~np.isnan(daily_rets)]

FTMO_ACCOUNT = 100_000
PROFIT_TARGET = 0.10
MAX_DAILY_LOSS = 0.05
MAX_TOTAL_LOSS = 0.10
TRADING_DAYS = 30
N_SIMULATIONS = 10_000

print(f'\nStrategy: RSI({b_rsi}) Top{b_topn} {b_rebal} Thresh={b_thresh}')
print(f'Observed daily returns: {len(daily_rets)} days')
print(f'  Mean daily return:    {np.mean(daily_rets)*100:.4f}%')
print(f'  Std daily return:     {np.std(daily_rets)*100:.4f}%')

np.random.seed(42)
equity_paths = np.zeros((N_SIMULATIONS, TRADING_DAYS + 1))
equity_paths[:, 0] = FTMO_ACCOUNT

passed = np.zeros(N_SIMULATIONS, dtype=bool)
blown = np.zeros(N_SIMULATIONS, dtype=bool)
daily_blown = np.zeros(N_SIMULATIONS, dtype=bool)
final_equity = np.zeros(N_SIMULATIONS)

for sim in range(N_SIMULATIONS):
    sim_rets = np.random.choice(daily_rets, size=TRADING_DAYS, replace=True)
    eq = FTMO_ACCOUNT
    for day in range(TRADING_DAYS):
        day_start = eq
        eq = eq * (1 + sim_rets[day])
        equity_paths[sim, day + 1] = eq
        if (eq - day_start) / FTMO_ACCOUNT < -MAX_DAILY_LOSS:
            daily_blown[sim] = True; equity_paths[sim, day+2:] = eq; break
        if (eq - FTMO_ACCOUNT) / FTMO_ACCOUNT < -MAX_TOTAL_LOSS:
            blown[sim] = True; equity_paths[sim, day+2:] = eq; break
        if (eq - FTMO_ACCOUNT) / FTMO_ACCOUNT >= PROFIT_TARGET:
            passed[sim] = True; equity_paths[sim, day+2:] = eq; break
    final_equity[sim] = equity_paths[sim, -1]

n_passed = passed.sum()
n_blown_total = blown.sum()
n_blown_daily = daily_blown.sum()
n_survived = N_SIMULATIONS - n_blown_total - n_blown_daily - n_passed
pass_rate = n_passed / N_SIMULATIONS * 100
mc_pass_rate = pass_rate

print(f'\n{"=" * 70}')
print(f'\U0001f3af MONTE CARLO RESULTS ({N_SIMULATIONS:,} simulations)')
print(f'{"=" * 70}')
print(f'  \u2705 PASSED:              {n_passed:>6,} ({n_passed/N_SIMULATIONS*100:.1f}%)')
print(f'  \u274c BLOWN (total DD):    {n_blown_total:>6,} ({n_blown_total/N_SIMULATIONS*100:.1f}%)')
print(f'  \u274c BLOWN (daily DD):    {n_blown_daily:>6,} ({n_blown_daily/N_SIMULATIONS*100:.1f}%)')
print(f'  \u23f3 Still trading:       {n_survived:>6,} ({n_survived/N_SIMULATIONS*100:.1f}%)')
print(f'  Median final equity:   ${np.median(final_equity):>10,.0f}')

verdict = '\U0001f7e2 FAVORABLE' if pass_rate >= 50 else '\U0001f7e1 POSSIBLE' if pass_rate >= 25 else '\U0001f7e0 CHALLENGING' if pass_rate >= 10 else '\U0001f534 UNLIKELY'
print(f'\n  FTMO VERDICT: {verdict} — {pass_rate:.1f}% pass rate')
print(f'{"=" * 70}')

# Monte Carlo plot
fig, axes = plt.subplots(2, 1, figsize=(16, 12), gridspec_kw={'height_ratios': [2, 1]})
fig.suptitle(f'FTMO Challenge Simulation | RSI Momentum Portfolio | {N_SIMULATIONS:,} Paths',
             fontsize=16, fontweight='bold')

sample_idx = np.random.choice(N_SIMULATIONS, min(300, N_SIMULATIONS), replace=False)
for i in sample_idx:
    c = '#2ca02c' if passed[i] else ('#d62728' if (blown[i] or daily_blown[i]) else '#aaaaaa')
    a = 0.3 if (passed[i] or blown[i] or daily_blown[i]) else 0.08
    axes[0].plot(range(TRADING_DAYS + 1), equity_paths[i], color=c, alpha=a, linewidth=0.5)
axes[0].axhline(FTMO_ACCOUNT * 1.10, color='green', linestyle='--', linewidth=2.5, label='10% Target ($110K)')
axes[0].axhline(FTMO_ACCOUNT * 0.90, color='red', linestyle='--', linewidth=2.5, label='10% Max Loss ($90K)')
axes[0].set_ylabel('Equity ($)'); axes[0].legend(fontsize=10)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[0].grid(True, alpha=0.2)

bins = np.linspace(final_equity.min(), final_equity.max(), 60)
axes[1].hist(final_equity, bins=bins, color='#1f77b4', alpha=0.7, edgecolor='white')
axes[1].axvline(np.median(final_equity), color='orange', linestyle='--', linewidth=2, label=f'Median: ${np.median(final_equity):,.0f}')
axes[1].axvline(FTMO_ACCOUNT * 1.10, color='green', linestyle='--', linewidth=2)
axes[1].axvline(FTMO_ACCOUNT * 0.90, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Final Equity ($)'); axes[1].set_ylabel('Frequency')
axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[1].legend(fontsize=10); axes[1].grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
# ════════════════════════════════════════════════════════════════
# EXPORT — JSON + CSV + PDF Tearsheet
# ════════════════════════════════════════════════════════════════

import json, datetime

STRATEGY_NAME = 'RSI_Momentum_Portfolio'
TICKER = 'PORTFOLIO'

EXPORT_DIR = 'strategy_exports'
try:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    EXPORT_DIR = '/content/drive/MyDrive/strategy_exports'
except:
    pass

out_dir = os.path.join(EXPORT_DIR, STRATEGY_NAME, TICKER)
os.makedirs(os.path.join(out_dir, 'latest'), exist_ok=True)
os.makedirs(os.path.join(out_dir, 'archive'), exist_ok=True)

run_id = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

summary = {
    'metadata': {
        'strategy': STRATEGY_NAME,
        'ticker': TICKER,
        'run_id': run_id,
        'start_date': str(close_df.index[0].date()),
        'end_date': str(close_df.index[-1].date()),
        'n_bars': len(close_df),
        'freq': FREQ,
        'universe_size': len(close_dict),
    },
    'params': {
        'rsi_period': b_rsi, 'top_n': b_topn,
        'rebal_freq': b_rebal, 'rsi_threshold': b_thresh,
        'lookback_rank': LOOKBACK_RANK,
    },
    'metrics': {
        'full': m_full, 'is': m_is, 'oos': m_oos, 'bh': m_bh,
    },
    'asset_attribution': asset_pct_sorted,
    'ftmo_monte_carlo': {
        'pass_rate': pass_rate, 'n_passed': int(n_passed),
        'n_blown_total': int(n_blown_total), 'n_blown_daily': int(n_blown_daily),
        'n_survived': int(n_survived),
    },
}

with open(os.path.join(out_dir, 'latest', 'summary.json'), 'w') as f:
    json.dump(summary, f, indent=2, default=str)

port_full.to_csv(os.path.join(out_dir, 'latest', 'daily_returns.csv'))

# Save dashboard as PDF
try:
    dashboard_fig.savefig(os.path.join(out_dir, 'latest', 'dashboard.pdf'),
                          dpi=150, bbox_inches='tight', facecolor='white')
    print(f'Dashboard PDF saved.')
except:
    print('Dashboard PDF save failed (try saving manually).')

print(f'\nExported to: {out_dir}/latest/')
print(f'Run ID: {run_id}')
print(f'Files: summary.json, daily_returns.csv, dashboard.pdf')